# 03 — Gold: dim_product

| Property | Value |
|----------|-------|
| **Gold Table** | `dim_product` |
| **Grain** | One row per ProductId |
| **Source** | `rpt.vwProduct` |
| **PK** | `ProductId` (int) |
| **Rows** | 31,565 |

**Hierarchy**: GlobalProductClass (16) → GlobalProductLine (73) → GlobalProduct (330) → Product (31,565)

**Cross-sell**: GlobalProductClass is the primary grouping for cross-sell matrix.

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *

LAKEHOUSE = "The_Global_Loom"
TABLE = "dim_product"
SOURCE_TABLE = "rpt.vwProduct"

print(f"Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

## Cell 2: Read silver source

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()

## Cell 3: Transform

- Rename `Product` → `ProductName` for clarity
- Drop ETL audit columns, SourceQuery, SourceKey, ParentKey, ParentId, IsMDSMappable
- Add "Unknown" member row for `-1` sentinel values

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================
df_clean = df_src.select(
    F.col("ProductId").cast("int"),
    F.col("ProductKey").cast("string"),
    F.col("Product").alias("ProductName").cast("string"),
    F.col("ProductDescription").cast("string"),
    F.col("GlobalProductClassId").cast("int"),
    F.col("GlobalProductClass").cast("string"),
    F.col("GlobalProductLineId").cast("int"),
    F.col("GlobalProductLine").cast("string"),
    F.col("GlobalProductId").cast("int"),
    F.col("GlobalProduct").cast("string"),
    F.col("DataSourceInstanceId").cast("int"),
    F.col("IsDeleted").cast("boolean")
)

# Filter out deleted Unknown members (ID = -1 AND IsDeleted = True)
before_count = df_clean.count()
df_clean = df_clean.filter(~((F.col("ProductId") == -1) & (F.col("IsDeleted") == True)))
after_count = df_clean.count()

if before_count > after_count:
    print(f"Filtered out {before_count - after_count} deleted Unknown member(s) (ID=-1, IsDeleted=True)")

print(f"After column select + filter: {after_count:,} rows × {len(df_clean.columns)} cols")
print(f"   Dropped: SourceQuery, SourceKey, ParentKey, ParentId, IsMDSMappable, ETL dates")

## Cell 4: Add "Unknown" member for sentinel value `-1`

When fact tables have `ProductId = -1`, this row catches them instead of breaking FK integrity.

In [ ]:
# ============================================================
# Cell 4: Add Unknown member
# ============================================================
unknown_row = spark.createDataFrame([(
    -1,             # ProductId
    "Unknown",      # ProductKey
    "Unknown",      # ProductName
    "Unknown",      # ProductDescription
    -1,             # GlobalProductClassId
    "Unknown",      # GlobalProductClass
    -1,             # GlobalProductLineId
    "Unknown",      # GlobalProductLine
    -1,             # GlobalProductId
    "Unknown",      # GlobalProduct
    -1,             # DataSourceInstanceId
    False           # IsDeleted
)], schema=df_clean.schema)

df_final = df_clean.unionByName(unknown_row)

print(f"Added Unknown member: {df_final.count():,} rows")

## Cell 5: Data quality checks

In [ ]:
# ============================================================
# Cell 5: Data quality checks
# ============================================================
total = df_final.count()
dupes = total - df_final.select("ProductId").distinct().count()
nulls = df_final.filter(F.col("ProductId").isNull()).count()
null_name = df_final.filter(F.col("ProductName").isNull()).count()
has_unknown = df_final.filter(F.col("ProductId") == -1).count()

print(f"DQ Checks")
print(f"   Total rows:       {total:,}")
print(f"   Duplicate PKs:    {dupes}")
print(f"   Null PKs:         {nulls}")
print(f"   Null names:       {null_name}")
print(f"   Unknown member:   {has_unknown} (expected 1)")

assert dupes == 0, f"ERROR: Found {dupes} duplicate ProductIds!"
assert nulls == 0, f"ERROR: Found {nulls} null ProductIds!"
assert has_unknown == 1, "ERROR: Missing Unknown member!"
print("\nAll DQ checks passed")

## Cell 6: Write to gold lakehouse

In [ ]:
# ============================================================
# Cell 6: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"Written: {TABLE}")
print(f"   Rows: {spark.table(TABLE).count():,}")